# Biomedical Literature Intelligence — GenAI + NLP over PubMed

**Author:** Eduardo Ramos Alves · [GitHub](https://github.com/eduramosalves) · [LinkedIn](https://www.linkedin.com/in/eduardo-alves-418757227)

A small, end-to-end pipeline that collects real biomedical literature and applies modern NLP + Generative AI to it. The domain is arbovirus surveillance (Dengue, Zika, Chikungunya) — the same area I researched as a CNPq scholar.

**What this notebook does:**
1. **Collects data** — fetches PubMed abstracts via the NCBI Entrez API.
2. **NLP — zero-shot classification** — labels each abstract by theme with a Hugging Face transformer, *no training data required*.
3. **NLP — biomedical NER** — extracts diseases, genes and chemicals.
4. **Generative AI** — summarizes each abstract and synthesizes a mini literature review (Hugging Face locally, with an optional OpenAI path).
5. **Validation** — hand-labels a held-out sample and measures classifier accuracy.
6. **Visualization** — topic distribution, top entities, and entity types.

**Skills demonstrated:** Python · Hugging Face `transformers` · Generative AI / LLMs · NLP · model validation · data collection · Pandas · Scikit-Learn · Matplotlib · runs on Google Colab (free tier).

> Tip: open in Colab and set Runtime → Change runtime type → **GPU** for faster inference (CPU also works).

## 1. Setup
Install dependencies. On Colab this takes ~1 minute.

In [ ]:
!pip install -q biopython transformers torch matplotlib pandas scikit-learn

import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline

pd.set_option('display.max_colwidth', 120)
print('Setup complete.')

## 2. Data collection — PubMed via NCBI Entrez
NCBI requires an email for API access. We pull abstracts on arbovirus surveillance and keep the title + abstract text.

In [ ]:
from Bio import Entrez

Entrez.email = 'eduardoramosalves23@gmail.com'  # required by NCBI

QUERY = '(dengue OR zika OR chikungunya) AND (genomic surveillance OR epidemiology OR diagnosis OR vaccine)'
N = 60

# 1) search for PubMed IDs
ids = Entrez.read(Entrez.esearch(db='pubmed', term=QUERY, retmax=N, sort='relevance'))['IdList']
print(f'Found {len(ids)} articles.')

# 2) fetch the records and parse title + abstract
records = Entrez.read(Entrez.efetch(db='pubmed', id=ids, rettype='xml'))

rows = []
for art in records['PubmedArticle']:
    cit = art['MedlineCitation']['Article']
    title = str(cit.get('ArticleTitle', ''))
    abs = cit.get('Abstract', {}).get('AbstractText', [])
    abstract = ' '.join(str(p) for p in abs).strip()
    if len(abstract) > 200:  # drop records with no usable abstract
        rows.append({'pmid': str(art['MedlineCitation']['PMID']), 'title': title, 'abstract': abstract})

df = pd.DataFrame(rows)
print(f'Kept {len(df)} abstracts with usable text.')
df.head(3)

## 3. NLP — zero-shot topic classification
Zero-shot lets us assign themes the model was never explicitly trained on. We define our own candidate labels and let the model score each abstract against them — no labeled training set needed.

In [ ]:
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

LABELS = ['epidemiology', 'diagnostics', 'vaccine', 'vector control',
          'genomics & sequencing', 'clinical treatment']

def top_topic(text):
    out = classifier(text[:1000], LABELS)  # truncate to keep inference fast
    return out['labels'][0], round(out['scores'][0], 3)

df[['topic', 'topic_score']] = df['abstract'].apply(lambda t: pd.Series(top_topic(t)))
df[['title', 'topic', 'topic_score']].head(8)

## 4. NLP — biomedical named-entity recognition (NER)
Extract structured entities (diseases, genes, chemicals) from free text using a domain-specific transformer.

In [ ]:
import re
from collections import Counter

ner = pipeline('token-classification', model='d4data/biomedical-ner-all',
               aggregation_strategy='max')

# This model frequently emits sub-word fragments ("den", "##ika", "z") instead
# of whole entities, which makes raw corpus counts unreadable. We clean each
# span and canonicalize the handful of arbovirus names that get split, so the
# section-7 chart shows real entities instead of token noise.
DISEASE_STEMS = {
    'dengue':       ('den', 'deng', 'denv', 'dengv'),
    'zika':         ('z', 'ika', 'zikv', 'ika virus'),
    'chikungunya':  ('chi', 'chik', 'chikv'),
    'yellow fever': ('yellow',),
    'arbovirus':    ('arbo', 'arbovir'),
}

def clean_entity(word):
    """Normalize a raw NER span to a canonical entity, or None to drop it."""
    w = re.sub(r'[^a-z ]', '', word.replace('##', '').lower()).strip()
    if not w:
        return None
    for canon, frags in DISEASE_STEMS.items():
        if canon in w:        # full name present as substring -> canonical form
            return canon
        if w in frags:        # known noisy fragment (exact match) -> canonical
            return canon
    return w if len(w) >= 4 else None   # drop leftover 1-3 char fragments

# Show the cleaned entities for one abstract.
sample = df.iloc[0]['abstract']
print('Sample abstract:', df.iloc[0]['title'], '\n')
seen = set()
for e in ner(sample[:1000]):
    w = clean_entity(e['word'])
    if w and e['score'] > 0.6 and (w, e['entity_group']) not in seen:
        seen.add((w, e['entity_group']))
        print(f"  {e['entity_group']:<22} {w:<28} ({e['score']:.2f})")

In [ ]:
# Corpus-level counts for the section-7 charts:
#   entity_counter -> cleaned & merged named entities (what)
#   type_counter   -> entity groups: Disease, Chemical, Gene... (what kind)
# Counting types is robust even when the model splits a name into sub-words.
entity_counter = Counter()
type_counter = Counter()

for text in df['abstract']:
    for e in ner(text[:1000]):
        if e['score'] <= 0.6:
            continue
        type_counter[e['entity_group']] += 1
        w = clean_entity(e['word'])
        if w:
            entity_counter[w] += 1

print('Top 15 named entities (cleaned & merged):')
for word, count in entity_counter.most_common(15):
    print(f'  {word:<28} {count}')

print('\nEntity types across the corpus:')
for grp, count in type_counter.most_common():
    print(f'  {grp:<28} {count}')

## 5. Generative AI — summarization + synthesis
First a local Hugging Face generative model condenses each abstract. Then we generate a short synthesized overview across the corpus. An optional OpenAI cell shows the same idea with a hosted LLM.

In [ ]:
# Some transformers builds (e.g. current Colab) don't register the 'summarization'
# pipeline task, so we load the seq2seq model directly and call generate() instead.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = 'sshleifer/distilbart-cnn-12-6'
tok = AutoTokenizer.from_pretrained(model_id)
mdl = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def summarize(text):
    inputs = tok(text[:1024], return_tensors='pt', truncation=True)
    ids = mdl.generate(**inputs, max_length=60, min_length=20, num_beams=4)
    return tok.decode(ids[0], skip_special_tokens=True)

df['summary'] = df['abstract'].apply(summarize)
df[['title', 'summary']].head(5)

In [ ]:
# OPTIONAL — higher-quality synthesis with an OpenAI model.
# Skips itself cleanly if no API key is set, so the notebook always runs.
import os

def synthesize_with_openai(summaries):
    from openai import OpenAI
    client = OpenAI()  # reads OPENAI_API_KEY from the environment
    joined = '\n'.join(f'- {s}' for s in summaries[:20])
    prompt = ('You are a biomedical research assistant. Given these abstract '
              'summaries on arbovirus surveillance, write a 4-sentence synthesis '
              f'of the main themes and gaps:\n\n{joined}')
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}])
    return resp.choices[0].message.content

if os.getenv('OPENAI_API_KEY'):
    print(synthesize_with_openai(df['summary'].tolist()))
else:
    print('No OPENAI_API_KEY set — skipping hosted-LLM synthesis.')
    print('Local fallback synthesis (concatenated top summaries):\n')
    print(' '.join(df['summary'].head(5)))

## 6. Validation
A model is only useful if we check it. I take a deterministic held-out sample, **read each abstract and hand-label it by PMID** (step 1 prints them, step 2 holds the labels), then measure how often the zero-shot classifier agrees. Labels are keyed by PMID rather than row position, so the score is correct even though PubMed's relevance ordering shifts between runs. This is the habit the IBM role asks for: *train, validate, and document under supervision.*

In [ ]:
# Step 1 — take a deterministic held-out sample and READ it.
# Sorting by PMID makes the sample stable across runs, so the labels you
# write below always line up with the right abstract (PubMed's relevance
# order is not stable, which is why positional labels are unreliable).
holdout = df.sort_values('pmid').head(15).reset_index(drop=True)

for _, r in holdout.iterrows():
    print(f"\nPMID {r['pmid']} — {r['title']}")
    print(r['abstract'][:600], '...')
    print(f"  >> model predicted: {r['topic']} ({r['topic_score']})")

print('\nChoose true labels from:', LABELS)

In [ ]:
# Step 2 — fill in the TRUE label for each PMID after reading it above.
# Keyed by PMID, so the scoring stays correct regardless of result ordering.
# Leave a PMID out and it's simply excluded from the metric (no fake numbers).
from sklearn.metrics import accuracy_score, classification_report

GROUND_TRUTH = {
    # 'PMID': 'epidemiology',   # <- copy a PMID from the print-out above and label it
}

labeled = holdout[holdout['pmid'].isin(GROUND_TRUTH)].copy()
if labeled.empty:
    print('No labels filled in yet.')
    print('Read the abstracts printed above, then add entries to GROUND_TRUTH')
    print('using one of these labels:', LABELS)
else:
    labeled['true_label'] = labeled['pmid'].map(GROUND_TRUTH)
    acc = accuracy_score(labeled['true_label'], labeled['topic'])
    print(f'Hand-labeled {len(labeled)}/{len(holdout)} abstracts.')
    print(f'Zero-shot accuracy on the labeled holdout: {acc:.0%}\n')
    print(classification_report(labeled['true_label'], labeled['topic'], zero_division=0))

## 7. Visualization
Three views of the corpus: topic distribution, the most frequent biomedical entities (cleaned and merged), and the breakdown of entity *types* (disease, chemical, gene…) — the last is robust even when the NER model splits a name into sub-word tokens.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

df['topic'].value_counts().plot(kind='barh', ax=axes[0], color='#1f9c9c')
axes[0].set_title('Abstracts by predicted topic')
axes[0].set_xlabel('count')

top = dict(entity_counter.most_common(10))
axes[1].barh(list(top.keys())[::-1], list(top.values())[::-1], color='#7c3aed')
axes[1].set_title('Top 10 biomedical entities')
axes[1].set_xlabel('mentions')

types = dict(type_counter.most_common(8))
axes[2].barh(list(types.keys())[::-1], list(types.values())[::-1], color='#ef6c4d')
axes[2].set_title('Entity types')
axes[2].set_xlabel('mentions')

plt.tight_layout()
plt.show()

## 8. Conclusion & next steps

This pipeline goes from raw PubMed text to classified, entity-tagged, and summarized output — combining classical NLP (zero-shot classification, NER) with Generative AI (abstractive summarization, LLM synthesis), validated against a hand-labeled sample.

**What I'd do next:**
- Replace the local summarizer with a RAG setup: embed abstracts (sentence-transformers) and answer free-text questions over the corpus.
- Fine-tune or few-shot the classifier on a larger hand-labeled set and report per-class F1.
- Add an AI-agent layer that, given a research question, plans the PubMed query, runs the pipeline, and returns a cited brief.
- Deploy as a small Streamlit app and containerize for a cloud runtime.

*Built by Eduardo Ramos Alves — biomedical scientist transitioning into data science & GenAI.*